In [1]:
import pandas as pd
from pathlib import Path

# Project paths
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load the integrated dataset created during Notebook 1
input_file = PROCESSED_DIR / "tourism_arrivals_monthly.csv"

tourism = pd.read_csv(input_file)

print("Dataset loaded successfully!")
print("Rows:", len(tourism))
print("Columns:", tourism.columns.tolist())

display(tourism.head())

Dataset loaded successfully!
Rows: 11280
Columns: ['Country', 'Month', 'Arrivals', 'Year', 'Month_Number', 'Date']


,Country,Month,Arrivals,Year,Month_Number,Date
0,Afghanistan,January,0.0,2022,1,2022-01-01
1,Albania,January,21.0,2022,1,2022-01-01
2,Algeria,January,6.0,2022,1,2022-01-01
3,Andorra,January,0.0,2022,1,2022-01-01
4,Angola,January,2.0,2022,1,2022-01-01


In [2]:
# ============================================================
# TOURISM DATA - COMPLETE CLEANING PIPELINE
# ============================================================

print("=" * 70)
print("STARTING DATA CLEANING")
print("=" * 70)

# 1. Remove unwanted "Country" header rows
before_rows = len(tourism)

tourism = tourism[
    tourism["Country"].astype(str).str.strip().str.lower() != "country"
].copy()

removed_rows = before_rows - len(tourism)

# 2. Clean text fields
tourism["Country"] = tourism["Country"].astype(str).str.strip()
tourism["Month"] = tourism["Month"].astype(str).str.strip()

# 3. Make sure Arrivals is numeric
tourism["Arrivals"] = pd.to_numeric(
    tourism["Arrivals"],
    errors="coerce"
)

# 4. Make sure Date is a proper datetime
tourism["Date"] = pd.to_datetime(tourism["Date"])

# 5. Preserve original country name
tourism["Country_Raw"] = tourism["Country"]

# 6. Create standardized country name
tourism["Country_Standardized"] = (
    tourism["Country"]
    .str.strip()
)

# 7. Reorder columns
tourism = tourism[
    [
        "Country_Raw",
        "Country_Standardized",
        "Year",
        "Month_Number",
        "Month",
        "Date",
        "Arrivals"
    ]
].copy()

# 8. Quality checks
missing_arrivals = tourism["Arrivals"].isna().sum()

negative_arrivals = (tourism["Arrivals"] < 0).sum()

duplicate_records = tourism.duplicated(
    subset=[
        "Country_Standardized",
        "Year",
        "Month_Number"
    ]
).sum()

# 9. Save cleaned dataset
output_file = PROCESSED_DIR / "tourism_arrivals_clean.csv"

tourism.to_csv(
    output_file,
    index=False
)

# ============================================================
# FINAL REPORT
# ============================================================

print("\n" + "=" * 70)
print("DATA CLEANING COMPLETE")
print("=" * 70)

print(f"Original rows:              {before_rows:,}")
print(f"Header rows removed:        {removed_rows:,}")
print(f"Final rows:                 {len(tourism):,}")
print(f"Unique countries:           {tourism['Country_Standardized'].nunique():,}")
print(f"Missing arrivals:            {missing_arrivals}")
print(f"Negative arrivals:          {negative_arrivals}")
print(f"Duplicate country-months:   {duplicate_records}")

print("\nYear coverage:")
display(
    tourism.groupby("Year")["Month_Number"]
    .nunique()
    .reset_index(name="Number_of_Months")
)

print("\nRemaining missing values:")
display(
    tourism[tourism["Arrivals"].isna()][
        [
            "Country_Standardized",
            "Year",
            "Month",
            "Arrivals"
        ]
    ]
)

print("\nFirst 5 rows:")
display(tourism.head())

print("\nClean dataset saved to:")
print(output_file)

print("\n" + "=" * 70)

STARTING DATA CLEANING

DATA CLEANING COMPLETE
Original rows:              11,280
Header rows removed:        170
Final rows:                 11,110
Unique countries:           229
Missing arrivals:            2
Negative arrivals:          0
Duplicate country-months:   0

Year coverage:


,Year,Number_of_Months
0,2022,12
1,2023,12
2,2024,12
3,2025,12
4,2026,10



Remaining missing values:


,Country_Standardized,Year,Month,Arrivals
1223,Guatemala,2022,July,NaN
3194,Papua New Guinea,2023,May,NaN



First 5 rows:


,Country_Raw,Country_Standardized,Year,Month_Number,Month,Date,Arrivals
0,Afghanistan,Afghanistan,2022,1,January,2022-01-01,0.0
1,Albania,Albania,2022,1,January,2022-01-01,21.0
2,Algeria,Algeria,2022,1,January,2022-01-01,6.0
3,Andorra,Andorra,2022,1,January,2022-01-01,0.0
4,Angola,Angola,2022,1,January,2022-01-01,2.0



Clean dataset saved to:
c:\DataAnalyticsPortfolio\01_Projects\01_Sri_Lanka_Tourism_Analytics\data\processed\tourism_arrivals_clean.csv

